In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip install -q transformers datasets torch scikit-learn pillow

In [ ]:
# ---- IMPORTS ----
import torch, numpy as np
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics import accuracy_score, f1_score
from torch.nn.functional import normalize

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# ---- HF LOGIN ----
from huggingface_hub import login
login(token="your_token")

In [ ]:
# ---- LOAD DATASET ----
from datasets import load_dataset
ds = load_dataset("Navarasa-9/Navarasa_Balanced")

README.md:   0%|          | 0.00/502 [00:00<?, ?B/s]

data/train-00000-of-00004.parquet:   0%|          | 0.00/420M [00:00<?, ?B/s]

data/train-00001-of-00004.parquet:   0%|          | 0.00/417M [00:00<?, ?B/s]

data/train-00002-of-00004.parquet:   0%|          | 0.00/425M [00:00<?, ?B/s]

data/train-00003-of-00004.parquet:   0%|          | 0.00/424M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/182M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6951 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/773 [00:00<?, ? examples/s]

In [ ]:
# columns: image, navarasa, text
labels = sorted(set(ds["train"]["navarasa"]))
label2id = {l: i for i, l in enumerate(labels)}

ds = ds.map(lambda x: {"label": label2id[x["navarasa"]]})

Map:   0%|          | 0/6951 [00:00<?, ? examples/s]

Map:   0%|          | 0/773 [00:00<?, ? examples/s]

In [ ]:
# ---- METRICS ----
def recall_at_k(sim, k):
    return sum(i in sim[i].topk(k).indices for i in range(sim.size(0))) / sim.size(0)

def retrieval_metrics(img_emb, txt_emb):
    sim = img_emb @ txt_emb.T
    return {
        "R@1": recall_at_k(sim, 1),
        "R@5": recall_at_k(sim, 5),
        "R@10": recall_at_k(sim, 10),
    }

# ---- MODEL ----
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

In [ ]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    images = [x["image"] for x in batch]
    texts  = [x["text"] for x in batch]
    labels = [x["label"] for x in batch]

    return images, texts, torch.tensor(labels)

In [ ]:
train_loader = DataLoader(
    ds["train"],
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)

In [ ]:
# ---- MODEL ----
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

In [ ]:
!pip install -q tqdm

In [ ]:
from tqdm import tqdm

In [ ]:
num_epochs = 25

model.train()
for epoch in range(num_epochs):
    epoch_loss = 0.0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{num_epochs}",
        leave=True
    )

    for step, (images, texts, _) in enumerate(progress_bar):
        inputs = processor(
            images=images,
            text=texts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        outputs = model(**inputs, return_loss=True)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()

        # 🔹 update tqdm bar
        progress_bar.set_postfix({
            "step": step,
            "loss": f"{loss.item():.4f}"
        })

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Avg Loss: {epoch_loss / len(train_loader):.4f}"
    )

Epoch 1/25: 100%|██████████| 435/435 [04:20<00:00,  1.67it/s, step=434, loss=0.2050]


Epoch [1/25] Avg Loss: 0.6909


Epoch 2/25: 100%|██████████| 435/435 [04:17<00:00,  1.69it/s, step=434, loss=0.4532]


Epoch [2/25] Avg Loss: 0.3624


Epoch 3/25: 100%|██████████| 435/435 [04:18<00:00,  1.68it/s, step=434, loss=0.0000]


...

Epoch [23/25] Avg Loss: 0.1445


Epoch 24/25: 100%|██████████| 435/435 [04:14<00:00,  1.71it/s, step=434, loss=0.1982]


Epoch [24/25] Avg Loss: 0.1354


Epoch 25/25: 100%|██████████| 435/435 [04:14<00:00,  1.71it/s, step=434, loss=0.2286]

Epoch [25/25] Avg Loss: 0.1344


In [ ]:
clip_save_dir = "/content/drive/MyDrive/models/clip-balanced-navarasa"

model.save_pretrained(clip_save_dir)
processor.save_pretrained(clip_save_dir)

print("CLIP model saved to Google Drive at:", clip_save_dir)

CLIP model saved to Google Drive at: /content/drive/MyDrive/models/clip-balanced-navarasa


In [ ]:
!zip -r /content/clip-balanced-navarasa.zip /content/drive/MyDrive/models/clip-balanced-navarasa

  adding: content/drive/MyDrive/models/clip-balanced-navarasa/ (stored 0%)
  adding: content/drive/MyDrive/models/clip-balanced-navarasa/config.json (deflated 66%)
  adding: content/drive/MyDrive/models/clip-balanced-navarasa/model.safetensors (deflated 7%)
  adding: content/drive/MyDrive/models/clip-balanced-navarasa/preprocessor_config.json (deflated 50%)
  adding: content/drive/MyDrive/models/clip-balanced-navarasa/tokenizer_config.json (deflated 63%)
  adding: content/drive/MyDrive/models/clip-balanced-navarasa/special_tokens_map.json (deflated 78%)
  adding: content/drive/MyDrive/models/clip-balanced-navarasa/vocab.json (deflated 62%)
  adding: content/drive/MyDrive/models/clip-balanced-navarasa/merges.txt (deflated 60%)
  adding: content/drive/MyDrive/models/clip-balanced-navarasa/tokenizer.json (deflated 83%)


In [ ]:
from google.colab import files

files.download("/content/clip-balanced-navarasa.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ---- EVALUATE ----
model.eval()
img_emb, txt_emb, y_true, y_pred = [], [], [], []

with torch.no_grad():
    for s in ds["test"]:
        img = processor(images=s["image"], return_tensors="pt").to(device)
        txt = processor(text=s["text"], return_tensors="pt", padding=True).to(device)

        i = normalize(model.get_image_features(**img), dim=-1)
        t = normalize(model.get_text_features(**txt), dim=-1)

        img_emb.append(i)
        txt_emb.append(t)

        y_pred.append((i @ t.T).argmax().item())
        y_true.append(s["label"])

img_emb = torch.cat(img_emb)
txt_emb = torch.cat(txt_emb)

print("\n[CLIP Evaluation]")
print("Retrieval:", retrieval_metrics(img_emb, txt_emb))
print("Accuracy:", accuracy_score(y_true, y_pred))
print("F1 Macro:", f1_score(y_true, y_pred, average="macro"))
print("F1 Weighted:", f1_score(y_true, y_pred, average="weighted"))



[CLIP Evaluation]
Retrieval: {'R@1': 0.17464424320827943, 'R@5': 0.592496765847348, 'R@10': 0.7943078913324709}
Accuracy: 0.11384217335058215
F1 Macro: 0.025551684088269456
F1 Weighted: 0.023270873995008663
